# Festa em Cena — análise exploratória de pedidos de Halloween

Projeto do Módulo 1 (SCTEC). Objetivo: ler, limpar e explorar `halloween_vendas.csv`
antes de comparar canais e períodos de venda.

In [3]:
#Bibliotecas usadas no projeto

import re                                               # Expressões regulares (limpeza de texto)
from datetime import datetime                           # Conversão de datas
from pathlib import Path                                # Caminho de arquivos
import pandas as pd                                     # Tabelas (DataFrames)

In [5]:
def ler_csv(caminho):
    """Lê o CSV e devolve um DataFrame com todas as colunas como texto.

    Lemos tudo como texto (dtype=str) para não deixar o Pandas
    adivinhar tipos: assim conseguimos ver valores problemáticos
    como eles vieram e convertê-los depois, com regras.
    """
    arquivo = Path(caminho)
    print("Pasta de execução:", Path.cwd())
    print("Arquivo encontrado:", arquivo.exists())
    df = pd.read_csv(arquivo, sep=",", encoding="utf-8", dtype=str)
    return df

In [6]:
CAMINHO_CSV = "halloween_vendas.csv" #Chama a função e guarda o que ela devolveu na variável

df_original = ler_csv(CAMINHO_CSV) #Usamos o nome “original” porque nunca vamos alterar essa tabela: o PDF exige preservar o original.

print("Total de linhas e colunas:", df_original.shape) # mostra (linhas, colunas). Antes de executar, tente prever: o CSV tem 1.501 linhas de texto, contando o cabeçalho. Quantas linhas o Pandas vai mostrar?
df_original.head(3) # Mostra os 3 primeiros registros, como o Sprint 1 pede.

Pasta de execução: c:\Users\N-lenovo\Documents\SCTEC\festa-em-cena
Arquivo encontrado: True
Total de linhas e colunas: (1500, 14)


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


Para cada coluna, isso mostra o nome, quantos valores estão preenchidos (non-null) e o tipo. Olhar com atenção para as colunas que têm menos preenchidos que o total de linhas: esses serão os nossos primeiros suspeitos de dados ausentes.

In [7]:
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   pedido_id       1500 non-null   str  
 1   data_venda      1500 non-null   str  
 2   produto         1500 non-null   str  
 3   categoria       1482 non-null   str  
 4   canal_venda     1500 non-null   str  
 5   cidade          1500 non-null   str  
 6   quantidade      1500 non-null   str  
 7   preco_unitario  1488 non-null   str  
 8   custo_unitario  1493 non-null   str  
 9   desconto_pct    1500 non-null   str  
 10  status_pedido   1500 non-null   str  
 11  publico         1500 non-null   str  
 12  tamanho         1500 non-null   str  
 13  data_entrega    1357 non-null   str  
dtypes: str(14)
memory usage: 164.2 KB


In [8]:
# Conta quantos valores vazios existem em cada coluna
ausentes = df_original.isna().sum()
print(ausentes)

pedido_id           0
data_venda          0
produto             0
categoria          18
canal_venda         0
cidade              0
quantidade          0
preco_unitario     12
custo_unitario      7
desconto_pct        0
status_pedido       0
publico             0
tamanho             0
data_entrega      143
dtype: int64


## Primeira inspeção

- A base tem 1.500 linhas e 14 colunas.
- Colunas com valores vazios: `categoria` (18), `preco_unitario` (12),
  `custo_unitario` (7) e `data_entrega` (143).
- O `info()` só detecta células vazias; valores preenchidos mas inválidos
  (ex.: quantidade 0 ou data escrita errada) serão investigados nos próximos passos.